# model.ipynb — Architecture CNN
BaseCNN (from scratch) + Transfer Learning (ResNet18 / DenseNet121 / EfficientNet-B0).

In [ ]:
from __future__ import annotations
from typing import Literal
import torch
import torch.nn as nn
from torchvision import models

print('Librairies importees.')

## 1. Bloc convolutif de base

In [ ]:
class ConvBlock(nn.Module):
    """
    Conv2d(k=3, pad=1) → BatchNorm2d → ReLU → MaxPool(2x2)
    Divise la résolution spatiale par 2.
    """
    def __init__(self, in_ch: int, out_ch: int, pool: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# Test
b = ConvBlock(3, 32)
x = torch.randn(2, 3, 224, 224)
print('ConvBlock sortie :', b(x).shape)  # (2, 32, 112, 112)

## 2. CNN Baseline complet

In [ ]:
class BaseCNN(nn.Module):
    """
    CNN 4 couches + GAP + MLP pour classification binaire NORMAL/PNEUMONIA.
    Paramètres : ~1.2M
    """
    def __init__(self, num_classes=1, dropout1=0.5, dropout2=0.3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),   # 224→112
            ConvBlock(32,  64),   # 112→56
            ConvBlock(64,  128),  # 56→28
            ConvBlock(128, 256),  # 28→14
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))  # 14→1
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout1),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# Vérification
m = BaseCNN()
x = torch.randn(4, 3, 224, 224)
print('Input :', x.shape, '-> Output :', m(x).shape)
n_params = sum(p.numel() for p in m.parameters())
print(f'Parametres totaux : {n_params:,}')

## 3. Transfer Learning

In [ ]:
class TransferModel(nn.Module):
    BACKBONES = ('resnet18', 'densenet121', 'efficientnet_b0')

    def __init__(self, backbone='resnet18', pretrained=False, freeze_backbone=True):
        super().__init__()
        if backbone == 'resnet18':
            base = models.resnet18(pretrained=pretrained)
            base.fc = nn.Linear(base.fc.in_features, 1)
        elif backbone == 'densenet121':
            base = models.densenet121(pretrained=pretrained)
            base.classifier = nn.Linear(base.classifier.in_features, 1)
        elif backbone == 'efficientnet_b0':
            base = models.efficientnet_b0(pretrained=pretrained)
            base.classifier[1] = nn.Linear(base.classifier[1].in_features, 1)
        else:
            raise ValueError(f'Backbone inconnu : {backbone}')
        self.model = base
        if freeze_backbone:
            for name, param in self.model.named_parameters():
                if 'fc' not in name and 'classifier' not in name:
                    param.requires_grad = False

    def forward(self, x):
        return self.model(x)

    def unfreeze(self):
        for param in self.model.parameters():
            param.requires_grad = True

# Test rapide (sans poids pretrained)
tm = TransferModel('resnet18', pretrained=False)
print('ResNet18 sortie :', tm(x).shape)
trainable = sum(p.numel() for p in tm.parameters() if p.requires_grad)
total     = sum(p.numel() for p in tm.parameters())
print(f'Parametres entrainables : {trainable:,} / {total:,}')

## 4. Factory `get_model()`

In [ ]:
def get_model(architecture: str = 'baseline', **kwargs) -> nn.Module:
    architecture = architecture.lower()
    if architecture == 'baseline':
        return BaseCNN(**kwargs)
    elif architecture in TransferModel.BACKBONES:
        return TransferModel(backbone=architecture, **kwargs)
    else:
        raise ValueError(f'Architecture inconnue : {architecture}')

# Exemples
print('Baseline :', get_model('baseline'))
print('Modele pret - OK')